# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khaled-dragon/ML-intern/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a "**classification**" task: predicting which discrete category a page
falls into (decline, stable, or growth) over a future window, based on its
prior performance signals. It's not ranking (I'm not just ordering pages
relative to each other for review capacity) or clustering (I'm not grouping
similar pages without labels) - I have a specific outcome I'm trying to
predict for each page.

### "#######################################################################"

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [2]:
!git clone https://github.com/khaled-dragon/ML-intern.git
%cd ML-intern
import pandas as pd


Cloning into 'ML-intern'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 136 (delta 48), reused 99 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 1.97 MiB | 5.28 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/ML-intern/ML-intern


In [3]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["trend_direction"].unique()

array(['down', 'stable', 'new', 'up', 'flat'], dtype=object)

For the real capstone, the target would be a **future outcome**: whether a
page's traffic declines, stays flat, or grows over the next 30 days, measured
from `fact_content_daily_performance` on the warehouse release. That needs
daily time-series data I don't have yet in the starter set.

As a stand-in for now, I'm using `trend_direction` from the starter data as a
**proxy label** — it's a snapshot classification, not a true future-window
label. I'll rebuild it properly with time windows once I have warehouse
access in Week 3.

### "#######################################################################"

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [4]:
df["trend_direction"].value_counts(normalize=True)

,proportion
trend_direction,
down,0.542067
stable,0.198733
up,0.146267
new,0.074533
flat,0.038400


I'd use **Precision@K** as my main metric (e.g., Precision@50), since the
real use case is a reviewer checking a limited number of top-ranked pages,
not classifying every page equally. I'd also track **recall on the decline
class** specifically, because missing a real decline (false negative) is more
costly than flagging a page that turns out fine (false positive).

### "#######################################################################"

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [5]:
cols = ["content_id", "client_id", "content_age_days", "impressions_90d", "sessions_90d", "avg_position", "ctr", "trend_direction"]
df[cols].head(1)


,content_id,client_id,content_age_days,impressions_90d,sessions_90d,avg_position,ctr,trend_direction
0,content_304f48230142,client_f369cb89fc,187,3803,17,10.6,0.76,down


One row = one content page, at one snapshot in time, for one client. In the
real capstone this becomes: one page, one client, one 90-day feature window
ending on a specific date, paired with a 30-day future outcome window that
starts right after it.

### "#######################################################################"

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [6]:
df.groupby("trend_direction")["content_age_days"].median()

,content_age_days
trend_direction,
down,216.0
flat,231.0
new,279.0
stable,300.0
up,291.5


A fixed rule like "flag pages older than X days as declining" would be wrong
here: declining pages actually have the *lowest* median age (216 days), not
the highest. The relationship isn't a clean threshold, it's counter-intuitive
and probably depends on interactions between several signals (age, position,
CTR, volume) at once, which is exactly the kind of pattern a fixed if-
statement can't capture, but a model trained on many features together can.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.